#### Ein Ensemble-Mitglied trainieren -- nur `SEED` unten aendern

Wie `01_agnews_qwen_lora_setup.ipynb`, aber alles, was sich pro Seed aendern wuerde (Zielordner-Name, Checkpoint-Dateiname), wird automatisch aus `SEED` abgeleitet -- kein manuelles Nachschauen/Umbenennen mehr noetig. Training laeuft live sichtbar (`!`-Magic, kein `subprocess.run(capture_output=True)`), damit du direkt siehst, ob's haengt oder einfach nur dauert.


In [1]:
# ============================================================
# SETUP ENVIRONMENT
# ============================================================

%cd /dss/dsshome1/00/ra58vit2/Masterarbeit/bayes_sub_inf

import os

uv_path = os.path.expanduser("~/.local/bin/uv")

if not os.path.exists(uv_path):
    print("Installing uv...")
    !curl -LsSf https://astral.sh/uv/install.sh | sh
else:
    print("uv already installed.")

!$HOME/.local/bin/uv sync --extra viz

print("SETUP FINISHED")


/dss/dsshome1/00/ra58vit2/Masterarbeit/bayes_sub_inf
uv already installed.


/dss/dsshome1/00/ra58vit2/Masterarbeit/bayes_sub_inf/.venv/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


Resolved 253 packages in 6ms
Checked 241 packages in 358ms
SETUP FINISHED


In [2]:
from pathlib import Path

baseline_script = r'''
"""Train a single-point Qwen2.5-0.5B LoRA baseline on AG News."""

import os
import argparse

import wandb
from jax import random

from subspace_inference.curve_optimizer.trainer.training_pipeline import (
    Config,
    train,
)
from subspace_inference.curve_optimizer.datasets.text_classification import (
    prepare_text_classification_splits,
)

WANDB_ENTITY = os.environ.get("WANDB_ENTITY")
WANDB_PROJECT = os.environ.get("WANDB_PROJECT", "curve_optimizer")


def train_qwen_lora_baseline(
    batch_size=4,
    num_steps=500,
    smoke_test=False,
    seed=2,
):

    print("current path:", os.getcwd())

    config_dict = {
        "rng_seed": seed,
        "load_params_path": False,

        "data": {
            "dataset_path": (
                "tests/qwen_data/helper_scripts/data_store/datasets/"
                "text_classification/ag_news"
            ),
            "val_percentage": 0.1,
        },

        "net_kwargs": {
            "model_type": "qwen",
            "model_path": (
                "tests/qwen_data/helper_scripts/data_store/models/"
                "qwen2.5_0.5B_float16"
            ),
        },

        "train_hyper": {
            "batch_size": batch_size,
            "num_epochs": -1,
            "num_steps": 10 if smoke_test else num_steps,
            "eval_every_n_batch": 200,
            "temperature": [1.0],
            "dataset_sampling": {"minibatch": {}},
            "save_params": True,
            "smoke_test": smoke_test,
        },

        "model_params": {
            # Single-point / standard fine-tuning
            "n_samples_eval": 1,
            "num_curve_segment": 0,
            "SegDeg": 1,
            "Pretraining": True,
            "curve_sampling_mode": "noBMA",

            # LoRA classification model
            "subspace_model": "lora_category",

            "natural_parameterization": False,
            "weight_decay": 0.0,

            "filter_masks": [
                {
                    "keys": ["self_attn", "q_proj", "kernel"],
                    "op": "all",
                },
                {
                    "keys": ["self_attn", "v_proj", "kernel"],
                    "op": "all",
                },
                {
                    "keys": ["lm_head", "kernel"],
                    "op": "all",
                },
            ],

            "lora_params": {
                "use_lora": True,
                "r": 8,
                "lora_dtype": "float32",
                "lora_alpha": 16.0,
                "lora_mode": "A(t)eB(t)",
                "rho_scheduler_frequency": 50.0,
                "lora_rho": 0.25,
                "lora_rho_s": 0.0,

                "filter_masks": [
                    {
                        "keys": ["self_attn", "q_proj", "kernel"],
                        "op": "all",
                        "dims": [1, 0],
                    },
                    {
                        "keys": ["self_attn", "v_proj", "kernel"],
                        "op": "all",
                        "dims": [1, 0],
                    },
                    {
                        "keys": ["lm_head", "kernel"],
                        "op": "all",
                        "dims": [1, 0],
                    },
                ],
            },
        },

        "optimizer_conf": {
            "name": "adamw",
            "kwargs": {
                "learning_rate": {
                    "name": "linear_onecycle_schedule",
                    "kwargs": {
                        "transition_steps": 10_000,
                        "peak_value": 1e-4,
                        "pct_start": 0.12,
                        "pct_final": 1.0,
                        "div_factor": 300.0,
                        "final_div_factor": 300.0,
                    },
                },
                "weight_decay": 0.001,
            },
            "freeze_other_params": True,
        },
    }

    logger = wandb.init(
        project=WANDB_PROJECT,
        name=f"agnews-qwen-lora-baseline-seed-{seed}",
        entity=WANDB_ENTITY,
        config=config_dict,
    )

    rng_key = random.PRNGKey(seed)

    data, unique_target_ids, rng_key = prepare_text_classification_splits(
        rng_key,
        dataset_path=config_dict["data"]["dataset_path"],
        val_percentage=config_dict["data"]["val_percentage"],
        batch_size=config_dict["train_hyper"]["batch_size"],
        smoke_test=config_dict["train_hyper"]["smoke_test"],
        logger=None,
    )

    config_dict["net_kwargs"]["target_token_ids"] = unique_target_ids

    config = Config.from_dict(
        config_dict,
        data,
    )

    env, params, config = train(
        logger,
        config,
        data,
    )

    print("AG News Qwen-LoRA baseline training successful!")

    wandb.finish()

    return env, params, config


if __name__ == "__main__":

    wandb.require("core")

    os.makedirs(
        "tmp_files",
        exist_ok=True,
    )

    parser = argparse.ArgumentParser()

    parser.add_argument(
        "--batch-size",
        type=int,
        default=4,
    )

    parser.add_argument(
        "--num-steps",
        type=int,
        default=500,
    )

    parser.add_argument(
        "--seed",
        type=int,
        default=2,
    )

    parser.add_argument(
        "--smoke-test",
        action="store_true",
    )

    args = parser.parse_args()

    train_qwen_lora_baseline(
        batch_size=args.batch_size,
        num_steps=args.num_steps,
        smoke_test=args.smoke_test,
        seed=args.seed,
    )
'''

path = Path(
    "experiments/ag_news_qwen_lora/train_agnews_qwen_lora_baseline.py"
)

path.parent.mkdir(
    parents=True,
    exist_ok=True,
)

path.write_text(
    baseline_script
)

print("Created:", path)


Created: experiments/ag_news_qwen_lora/train_agnews_qwen_lora_baseline.py


### CONFIG -- **nur diese Zelle** pro Ensemble-Mitglied aendern


In [3]:
# ============================================================
# CONFIG -- einziger Ort, den du pro Mitglied aendern musst
# ============================================================

SEED = 1

BATCH_SIZE = 4
NUM_STEPS = 500

print("Training member with SEED =", SEED)


Training member with SEED = 1


In [4]:
# ============================================================
# TRAIN (live sichtbarer Output ueber !-Magic)
# ============================================================

from pathlib import Path

TMP_FILES_DIR = Path("tmp_files")
TMP_FILES_DIR.mkdir(parents=True, exist_ok=True)

files_before = set(TMP_FILES_DIR.glob("*_pretrained_params.npy"))

train_cmd = (
    "WANDB_MODE=offline $HOME/.local/bin/uv run python "
    "experiments/ag_news_qwen_lora/train_agnews_qwen_lora_baseline.py "
    f"--batch-size {BATCH_SIZE} "
    f"--num-steps {NUM_STEPS} "
    f"--seed {SEED}"
)
print("Running:", train_cmd)
!{train_cmd}

# ------------------------------------------------------------
# Neue Checkpoint-Datei automatisch erkennen (Dateiliste vorher/
# nachher vergleichen -- Dateiname enthaelt eine zufaellige wandb-
# Run-ID, kein manuelles Nachschauen noetig).
# ------------------------------------------------------------
files_after = set(TMP_FILES_DIR.glob("*_pretrained_params.npy"))
new_files = files_after - files_before

assert len(new_files) == 1, (
    f"Erwartet genau 1 neue Checkpoint-Datei, gefunden: {len(new_files)} "
    f"({new_files}). Training vermutlich fehlgeschlagen -- Output oben pruefen."
)

new_params_path = new_files.pop()
new_metrics_path = new_params_path.with_name(
    new_params_path.name.replace(
        "_pretrained_params.npy", "_cp_fixed_0_metrics.npz",
    )
)

assert new_metrics_path.exists(), f"Metrics-Datei fehlt: {new_metrics_path}"

print()
print("Neuer Checkpoint gefunden:")
print(" params: ", new_params_path)
print(" metrics:", new_metrics_path)


Running: WANDB_MODE=offline $HOME/.local/bin/uv run python experiments/ag_news_qwen_lora/train_agnews_qwen_lora_baseline.py --batch-size 4 --num-steps 500 --seed 1
current path: /dss/dsshome1/00/ra58vit2/Masterarbeit/bayes_sub_inf
wandb: Using wandb-core as the SDK backend. Please refer to https://wandb.me/wandb-core for more information.
wandb: Tracking run with wandb version 0.17.9
wandb: W&B syncing is set to `offline` in this directory.  
wandb: Run `wandb online` or set WANDB_MODE=online to enable cloud syncing.
Dataset loaded: 120000 samples, 4 classes
Target token IDs: [[15]
 [16]
 [17]
 [18]]
Validation data are used
Training data: 108000 samples, Validation data: 12000 samples
Computed cp_fix: [1]
Setting num_epochs to 1 to match 500 steps
Pretraining fixed control points
Using cosine rho schedule with total steps 500
save best parameters
Training cp member 0
Number of trainable parameters: 540_672
Start minibatch training for 1 epochs
post_predict_logits shape: (12000, 4)
Run

In [5]:
# ============================================================
# SAVE (Zielordner-Name automatisch aus SEED abgeleitet)
# ============================================================

import shutil

CHECKPOINT_DIR = (
    Path.home() / "Masterarbeit" / "checkpoints" / f"qwen_lora_agnews_seed{SEED}"
)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

shutil.copy(new_params_path, CHECKPOINT_DIR / new_params_path.name)
shutil.copy(new_metrics_path, CHECKPOINT_DIR / new_metrics_path.name)

print("Gespeichert nach:", CHECKPOINT_DIR)
for p in sorted(CHECKPOINT_DIR.iterdir()):
    print(f"  {p.name}  ({p.stat().st_size / 1024:.1f} KB)")


Gespeichert nach: /dss/dsshome1/00/ra58vit2/Masterarbeit/checkpoints/qwen_lora_agnews_seed1
  iiljya0g_cp_fixed_0_metrics.npz  (334.5 KB)
  iiljya0g_pretrained_params.npy  (2150.7 KB)
